In [ ]:
##%% Feature Availability Flags
# Configuration for optional notebook functionality
# Set these flags to enable/disable features based on your requirements

# Prediction and modeling capabilities
HAVE_FINANCE_PREDICTION = True  # Enable financial prediction features

# Database connectivity
HAVE_DATABASE_CONNECTION = False  # Enable PostgreSQL database connections

# Advanced analytics and visualizations
HAVE_ADVANCED_ANALYTICS = True  # Enable advanced analytics features

# Dimensionality reduction visualizations
HAVE_DIM_REDUCTION = False  # Enable dimensionality reduction visualizations

# Debug and development features
DEBUG_MODE = False  # Enable debug output and additional logging

# Optional feature toggles
ENABLE_SECTOR_ANALYSIS = True  # Enable sector-based analysis
ENABLE_REGION_ANALYSIS = True  # Enable region-based analysis
ENABLE_INTERACTIVE_PLOTS = True  # Enable interactive Plotly visualizations
ENABLE_EXCEL_EXPORT = True  # Enable Excel export functionality

print("=" * 80)
print("FEATURE FLAGS CONFIGURATION")
print("=" * 80)
print(f"Financial Prediction:        {HAVE_FINANCE_PREDICTION}")
print(f"Database Connection:         {HAVE_DATABASE_CONNECTION}")
print(f"Advanced Analytics:          {HAVE_ADVANCED_ANALYTICS}")
print(f"Dimensionality Reduction:    {HAVE_DIM_REDUCTION}")
print(f"Debug Mode:                  {DEBUG_MODE}")
print(f"Sector Analysis:             {ENABLE_SECTOR_ANALYSIS}")
print(f"Region Analysis:             {ENABLE_REGION_ANALYSIS}")
print(f"Interactive Plots:           {ENABLE_INTERACTIVE_PLOTS}")
print(f"Excel Export:                {ENABLE_EXCEL_EXPORT}")
print("=" * 80)

## Configuration

Load configuration from environment variables or config files.


## Sample Data Generator

Create sample financial dataset for demonstration when real data is unavailable.

Note: The generator is now provided by the package as
`finance_ml.create_sample_financial_dataset` — no inline definition needed here.


In [ ]:
# Load stock data using package strategy helpers
all_stocks = load_stock_data(config)
if all_stocks is None or len(all_stocks) == 0:
    raise ValueError("Failed to load any stock data")

display_data_summary(all_stocks)


In [ ]:
# Unified validation reporting
validate_and_display_data(all_stocks)


In [ ]:
# Unified EDA display
perform_and_display_eda(all_stocks)


In [ ]:
# Preprocess data
try:
    all_stocks_processed = preprocess(all_stocks)
    logger.info(f"Data preprocessed: {all_stocks_processed.shape}")
    print(f"\n✓ Data preprocessed successfully")
    print(f"  Shape after preprocessing: {all_stocks_processed.shape}")
except Exception as e:
    logger.error(f"Preprocessing failed: {e}")
    print(f"✗ Preprocessing failed: {e}")
    all_stocks_processed = all_stocks.copy()


In [ ]:
# Build features and target
try:
    X, y, numeric_features, categorical_features = build_features_and_target(all_stocks_processed)
    logger.info(f"Features built: {X.shape}, Target: {y.shape if y is not None else 'None'}")
    print(f"\n✓ Features engineered successfully")
    print(f"  Feature matrix shape: {X.shape}")
    print(f"  Target shape: {y.shape if y is not None else 'None'}")
    print(f"  Numeric features: {len(numeric_features)}")
    print(f"  Categorical features: {len(categorical_features)}")
    print(f"  First 10 numeric features: {numeric_features[:10]}")
    if categorical_features:
        print(f"  Categorical features: {categorical_features}")
except Exception as e:
    logger.error(f"Feature engineering failed: {e}")
    print(f"✗ Feature engineering failed: {e}")
    raise

In [ ]:
# Create event labels for classification
try:
    event_labels = create_event_labels(all_stocks_processed)
    print(f"\n✓ Event labels created")
    print(f"  Label distribution: {pd.Series(event_labels).value_counts().to_dict()}")

    # Train event classifier using the DataFrame with proper feature separation
    classifier_results = train_event_classifier(all_stocks_processed, event_labels)
    print(f"\n✓ Event classifier trained")
    print(f"  Accuracy: {classifier_results.get('accuracy', 0):.4f}")
    print(f"  F1 Score (macro): {classifier_results.get('f1_macro', 0):.4f}")
except Exception as e:
    logger.error(f"Classification training failed: {e}")
    print(f"✗ Classification training failed: {e}")

In [ ]:
# Train baseline regression model
try:
    # Use the proper training function with required parameters
    from pathlib import Path

    output_dir = Path(config.output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    regression_results = train_and_evaluate_regression(
            all_stocks_processed,
            out_dir=output_dir,
            n_jobs=config.n_jobs if hasattr(config, 'n_jobs') else -1
            )

    if regression_results:
        print(f"\n✓ Regression model trained")
        print(f"  MAE: {regression_results.get('mae', 0):.4f}")
        print(f"  RMSE: {regression_results.get('rmse', 0):.4f}")
        print(f"  R²: {regression_results.get('r2', 0):.4f}")

        # Store predictions in dataframe for later use
        if 'predictions' in regression_results:
            pred_df = regression_results['predictions']
            all_stocks_processed.loc[pred_df.index, 'predicted_price_target'] = pred_df['y_pred'].values
    else:
        print("⚠ Regression training skipped (insufficient data or dry run)")
except Exception as e:
    logger.error(f"Regression training failed: {e}")
    print(f"✗ Regression training failed: {e}")

In [ ]:
# Calculate mispricing scores
try:
    # Get predictions from regression model
    if 'predictions' in regression_results:
        predictions = regression_results['predictions']
        all_stocks_processed['predicted_price_target'] = predictions

        # Calculate mispricing
        mispricing = calculate_mispricing_score(all_stocks_processed)
        all_stocks_processed['mispricing_score'] = mispricing

        print(f"\n✓ Mispricing scores calculated")
        print(f"  Mean mispricing: {mispricing.mean():.4f}")
        print(f"  Std mispricing: {mispricing.std():.4f}")

        # Rank undervalued stocks
        undervalued = rank_undervalued_stocks(all_stocks_processed, top_n=10)
        print(f"\nTop 10 Undervalued Stocks:")
        print(undervalued[['ticker', 'name', 'sector', 'mispricing_score']].to_string())

        # Rank overvalued stocks
        overvalued = rank_overvalued_stocks(all_stocks_processed, top_n=10)
        print(f"\nTop 10 Overvalued Stocks:")
        print(overvalued[['ticker', 'name', 'sector', 'mispricing_score']].to_string())

except Exception as e:
    logger.error(f"Valuation analysis failed: {e}")
    print(f"✗ Valuation analysis failed: {e}")

In [ ]:
# Demonstrate proper preprocessing pipeline with separate transformers
try:
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split

    if y is not None and len(X) > 0:
        print("\n" + "=" * 80)
        print("ADVANCED PREPROCESSING PIPELINE DEMONSTRATION")
        print("=" * 80)

        # Show feature type separation
        print(f"\n📊 Feature Type Analysis:")
        print(f"  Total features: {len(numeric_features) + len(categorical_features)}")
        print(f"  Numeric features: {len(numeric_features)}")
        print(f"  Categorical features: {len(categorical_features)}")

        # Build preprocessing pipeline with separate transformers
        print(f"\n🔧 Building preprocessing pipeline with separate transformers...")

        preprocessor = ColumnTransformer(
                transformers=[
                    ('numeric', StandardScaler(with_mean=False), numeric_features),
                    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
                    ],
                remainder='drop'
                )

        # Create full pipeline with regressor
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
            ])

        # Prepare data (remove NaN targets)
        mask = ~y.isna()
        X_clean = X.loc[mask]
        y_clean = y.loc[mask]

        if len(X_clean) >= 20:
            # Split data
            X_train, X_test, y_train, y_test = train_test_split(
                    X_clean, y_clean, test_size=0.2, random_state=42
                    )

            print(f"\n📈 Training pipeline...")
            print(f"  Training samples: {len(X_train)}")
            print(f"  Test samples: {len(X_test)}")

            # Fit pipeline
            pipeline.fit(X_train, y_train)

            # Make predictions
            y_pred = pipeline.predict(X_test)

            # Calculate metrics
            from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)

            print(f"\n✓ Pipeline training completed")
            print(f"  MAE: {mae:.4f}")
            print(f"  RMSE: {rmse:.4f}")
            print(f"  R²: {r2:.4f}")

            # Show transformed feature dimensions
            X_transformed = preprocessor.transform(X_test)
            print(f"\n🔄 Transformed feature space:")
            print(f"  Original features: {X_test.shape[1]}")
            print(f"  Transformed features: {X_transformed.shape[1]}")
            print(
                    f"  (Numeric: {len(numeric_features)}, One-Hot Encoded Categorical: {X_transformed.shape[1] - len(numeric_features)})")

            print(f"\n✓ Preprocessing pipeline demonstration complete")
        else:
            print(f"\n⚠ Insufficient clean data ({len(X_clean)} samples) for pipeline demo")
    else:
        print("\n⚠ No target variable or features available for pipeline demo")

except Exception as e:
    logger.error(f"Pipeline demo failed: {e}")
    print(f"✗ Pipeline demo failed: {e}")
    import traceback

    traceback.print_exc()

In [ ]:
# Create sector heatmap
try:
    if 'sector' in all_stocks_processed.columns and 'mispricing_score' in all_stocks_processed.columns:
        fig = create_sector_heatmap(all_stocks_processed)
        print("\n✓ Sector heatmap created")
except Exception as e:
    logger.error(f"Sector heatmap failed: {e}")
    print(f"✗ Sector heatmap creation failed: {e}")

# Create interactive prediction plot
try:
    if 'predicted_price_target' in all_stocks_processed.columns and 'price_target' in all_stocks_processed.columns:
        fig = create_interactive_prediction_plot(all_stocks_processed)
        print("✓ Interactive prediction plot created")
except Exception as e:
    logger.error(f"Prediction plot failed: {e}")
    print(f"✗ Prediction plot creation failed: {e}")

print("\n" + "=" * 80)
print("Analysis complete!")
print("=" * 80)

## Key Improvements: Proper Preprocessing Pipelines

This notebook now uses the returned `numeric_features` and `categorical_features` lists from `build_features_and_target()` to create proper preprocessing pipelines with:

1. **Separate Transformers**: 
   - `StandardScaler` for numeric features
   - `OneHotEncoder` for categorical features

2. **Benefits**:
   - Proper handling of different feature types
   - Prevents data leakage by fitting transformers only on training data
   - Automatically handles unknown categories in test data
   - Cleaner, more maintainable code

3. **Implementation**:
   - `build_features_and_target()` returns 4 values: `X, y, numeric_features, categorical_features`
   - These lists are used in `ColumnTransformer` for proper preprocessing
   - All model training functions now use this pattern

See the "Advanced Preprocessing Pipeline Demonstration" section above for a working example.